In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [47]:
elections_df = pd.read_csv("/Users/ephzel/Downloads/GitHubLocal/qtm151fall2025/Final Project Files/4-US_Election_2020/president_county.csv")
print("States:", elections_df['state'].unique()) #we see that all 50 states are included + DC

# Top county per state by number of votes
elections_df_sorted = elections_df.sort_values(
    by=["state", "total_votes"],
    ascending=[True, False]
)

elections_df_sorted["state_rank"] = elections_df_sorted.groupby("state")["total_votes"] \
                                                       .rank(method="first", ascending=False)
#create state rank to allow most populous counties to 'rise to the top', easier for us to work with
top_county_per_state = elections_df_sorted[elections_df_sorted["state_rank"] == 1]

# Finding county level winners
pcc_df = pd.read_csv("/Users/ephzel/Downloads/GitHubLocal/qtm151fall2025/Final Project Files/4-US_Election_2020/president_county_candidate.csv")
county_winner = (
    pcc_df.groupby(["state", "county", "candidate"])["total_votes"]
          .sum()
          .reset_index()
)

# Keep the candidate with the highest votes in each county
county_winner = county_winner.loc[
    county_winner.groupby(["state", "county"])["total_votes"].idxmax()
]

county_winner = county_winner.rename(columns={
    "candidate": "county_winner_candidate",
    "votes": "county_winner_votes"
}) # that way we can compare this to the overall state winner, create a boolean to determine if the most populous county is in fact, a bellwether

# State-level winner
state_winner = (
    pcc_df.groupby(["state", "candidate"])["total_votes"]
          .sum()
          .reset_index()
)

# Keep the top candidate per state
state_winner = state_winner.loc[
    state_winner.groupby("state")["total_votes"].idxmax()
]

state_winner = state_winner.rename(columns={
    "candidate": "state_winner_candidate",
    "votes": "state_winner_votes"
})


# Merge winners into top_county_per_state
merged_top_county_per_state = top_county_per_state.merge(
    county_winner,
    on=["state", "county"],
    how="left"
)

merged_top_county_per_state = merged_top_county_per_state.merge(
    state_winner,
    on="state",
    how="left"
)

# Bellwether indicator?
merged_top_county_per_state["bellwether"] = (
    merged_top_county_per_state["county_winner_candidate"] == merged_top_county_per_state["state_winner_candidate"]
)
# Results
display(merged_top_county_per_state)
print("Number of bellwether counties:", merged["bellwether"].sum())
print("Total states analyzed:", merged.shape[0])
print("Bellwether accuracy:", merged["bellwether"].mean())

States: ['Delaware' 'District of Columbia' 'Florida' 'Georgia' 'Hawaii' 'Idaho'
 'Illinois' 'Indiana' 'Iowa' 'Kansas' 'Kentucky' 'Louisiana' 'Maine'
 'Maryland' 'Massachusetts' 'Michigan' 'Minnesota' 'Mississippi'
 'Missouri' 'Montana' 'Nebraska' 'Nevada' 'New Hampshire' 'New Jersey'
 'New Mexico' 'New York' 'North Carolina' 'North Dakota' 'Ohio' 'Oklahoma'
 'Oregon' 'Pennsylvania' 'Rhode Island' 'South Carolina' 'South Dakota'
 'Tennessee' 'Texas' 'Utah' 'Vermont' 'Virginia' 'Washington'
 'West Virginia' 'Wisconsin' 'Wyoming' 'Alabama' 'Alaska' 'Arkansas'
 'California' 'Colorado' 'Connecticut' 'Arizona']


,state,county,current_votes,total_votes_x,percent,state_rank,county_winner_candidate,total_votes_y,state_winner_candidate,total_votes,bellwether
0,Alabama,Jefferson County,325848,325848,100,1.0,Joe Biden,181688,Donald Trump,1441168,False
1,Alaska,ED 28,25580,25580,100,1.0,Write-ins,12831,Donald Trump,189892,False
2,Arizona,Maricopa County,2069475,2068144,100,1.0,Joe Biden,1040774,Joe Biden,1672143,True
3,Arkansas,Pulaski County,169956,169956,100,1.0,Joe Biden,101947,Donald Trump,760647,False
4,California,Los Angeles County,4263443,4263443,100,1.0,Joe Biden,3028885,Joe Biden,11109764,True
5,Colorado,Denver County,393827,393827,100,1.0,Joe Biden,313293,Joe Biden,1804352,True
6,Connecticut,Stamford,59412,59412,100,1.0,Joe Biden,40437,Joe Biden,1080680,True
7,Delaware,New Castle County,287633,287633,100,1.0,Joe Biden,195034,Joe Biden,296268,True
8,District of Columbia,Ward 6,62918,62918,100,1.0,Joe Biden,56719,Joe Biden,317323,True
9,Florida,Miami-Dade County,1156816,1156816,100,1.0,Joe Biden,617864,Donald Trump,5668731,False


Number of bellwether counties: 34
Total states analyzed: 51
Bellwether accuracy: 0.6666666666666666
